In [1]:
# подключаем библиотеки

# библиотека для отправки запроса
import requests

# импортируем конструктор BeautifulSoup для преобразования полученного текста
from bs4 import BeautifulSoup

import pandas as pd
import numpy as np
import time

В качестве источника информации возьмем сайт издательства [Питер](https://www.piter.com/),
раздел [Компьютерная литература](https://www.piter.com/collection/kompyutery-i-internet).

Данный раздел имеет несколько страниц -- на момент написания материала 44 страницы.
Обратим внимание также на адреса страниц
- [1-я страница](https://www.piter.com/collection/kompyutery-i-internet?page=1) --- https://www.piter.com/collection/kompyutery-i-internet?page=1;
- [2-я страница](https://www.piter.com/collection/kompyutery-i-internet?page=2) --- https://www.piter.com/collection/kompyutery-i-internet?page=2
...

Имеем одинаковый адрес страниц, отличающийся только номером страницы. Воспользуемся этим фактом для обработки всех страниц.

In [ ]:
# Запишем адрес страницы парсинга в отдельную переменную.
# В общем случае этого можно не делать
# C помощью метода `get` оправляем запрос к выбранному ресурсу
# Ответ записываем в переменную

link = 'https://www.piter.com/collection/kompyutery-i-internet?page=1'
answer = requests.get(link)

In [ ]:
# Проверим код завершения операции. Желательно, чтобы он был равен 200 -- получен ответ на запрос.
# Напечатаем полученный текст запроса
answer.status_code
answer.text;

In [ ]:
# с помощью конструктора BeautifulSoup и парсера `lxml` размещаем полученную строку
# и результат записываем в переменную.

soup = BeautifulSoup(answer.text, 'lxml')
soup.prettify();

In [ ]:
# Исследуем страницу и найдем теги, которые содержат всю информацию по интересующему объекту -- здесь это книги.
# Для поиска тегов применяется следующая последовательность действий:
# 1. На странице выделяем какой-либо элемент у объкта, например, у книги это может быть автор, название и т.п.;
# 2. Щелкаем правой кнопкой и выбираем "Просмотреть код" (в зависимости от браузера надпись может быть немного другой);
# 3. Справа откроется Просмотровщик кода и выделится фрагмент кода, соответствующий выделенному фрагменту на странице;
# 4. Двигаясь в Просмотровщике от выделенного фрагмента кода, чаще всего вверх, находим строку кода, которая
# выделяет полностью информацию об объекте.
# 5. Передаем название соответствующего тега и дополнительную информацию методам `find_all` и `find`.


# метод find_all -- находит все элементы с заданным тегом -- возвращает список
# метод find -- находит только первый элемент с заданным тегом -- возвращает элемент с искомыми тегами

# В данном случае тегом, который содержит информацию о книге является тег `div`.
# Однако, если воспользоваться командой
# soup.find_all('div')
# то он вернет все теги div. Для того, чтобы вернуть требуемые теги у методов `find_all` и `find`
# есть параметр `attrs`. Назначение этого параметра -- подсказать какие именно теги надо искать
# за счет указания дополнительной информации.

list_books = soup.find_all('div', attrs={'class': ['grid-3 prod-block book-block',
                                                  'grid-3 prod-block book-block clear-class2 clear-class4',
                                                  'grid-3 prod-block book-block clear-class2']
                                        }
                          )

# Самое сложное -- это найти нужные теги.

In [ ]:
# Далее необходимо понять как представлена информация об объекте с помощью кода
# Здесь лучше исследовать одну запись, что понять механизм представления.
list_books[0] # выводим информацию о первом объекте -- книге

# Замечаем, что
# -- автор указан здесь <span class="author">Грегг Б. </span>
# -- название здесь <span class="title">Производительность систем</span>
# -- ссылка здесь <a href="/collection/kompyutery-i-internet/product/proizvoditelnost-sistem" title="Производительность систем">
# -- цена здесь <span class="price">5199 р.</span>

#Попробуем извлечь эту информацию

<div class="grid-3 prod-block book-block clear-class2 clear-class4">
<a href="/collection/kompyutery-i-internet/product/proizvoditelnost-sistem" title="Производительность систем">
<span class="img">
<img alt="Производительность систем" class="coverProduct" src="https://static.insales-cdn.com/images/products/1/4840/662967016/large_44611818.jpg"/>
</span>
<span class="author">Грегг Б. </span>
<span class="title">Производительность систем</span>
</a>
<div class="buyzone clearfix" style="height: 86px;">
<div class="grid-7 button_block_wrapper">
<form action="/cart_items" class="addToCart" method="post">
<input name="variant_id" type="hidden" value="603024977"/>
<button class="buy">Купить</button><!-- !1 -->
</form>
</div>
<div class="grid-5 padded-left">
<span class="prices">
<span class="price">5199 р.</span>
</span>
</div>
<span class="cat-prod-digit-count" style="display: none;">
<i>2</i>
</span>
<div class="grid-12 catalog-digit-count">
<div class="grid-7">
<!--<button onclick="locatio

In [ ]:
list_books[0].find('span', attrs={'class': 'author'}).get_text(strip=True) #забираем автора
list_books[0].find('span', attrs={'class': 'title'}).get_text(strip=True) #забираем название
'https://www.piter.com' + list_books[0].find('a').get('href') #Забираем ссылку + добавляем недостающую часть адреса
price = list_books[0].find_all('div', attrs={'class': 'grid-5 padded-left'}) #рабираем сразу две цены
price[0].find('span', attrs={'class': 'price'}).get_text(strip=True) #берем первую цену

С одной записью list_books[0] справились. Теперь обработаем все.

In [ ]:
#создаем структуру для хранения сибираемых данных
dict_df = {
    'author': [],
    'title': [],
    'url_book': [],
    'price': []
          }

for book in list_books:# перебиараем записи (книги). Последующий код взят выше.
    dict_df['author'].append(book.find('span', attrs={'class': 'author'}).get_text(strip=True))
    dict_df['title'].append(book.find('span', attrs={'class': 'title'}).get_text(strip=True))
    dict_df['url_book'].append('https://www.piter.com' + book.find('a').get('href'))
    price = book.find_all('div', attrs={'class': 'grid-5 padded-left'})
    pr = [price[i].find('span', attrs={'class': 'price'}).get_text(strip=True) for i in range(0,len(price))]
    dict_df['price'].append(' / '.join(pr))

In [ ]:
pd.DataFrame(dict_df) # генерируем датафрейм

,author,title,url_book,price
0,Грегг Б.,Производительность систем,https://www.piter.com/collection/kompyutery-i-...,5199 р. / 799 р.
1,Апельцин Л.,Data Science в действии,https://www.piter.com/collection/kompyutery-i-...,5029 р. / 699 р.
2,Одерски М.,Scala. Профессиональное программирование. 4-е ...,https://www.piter.com/collection/kompyutery-i-...,4436 р. / 799 р.
3,Ромеро Б.,Игровой баланс. Точная наука геймдизайна,https://www.piter.com/collection/kompyutery-i-...,4115 р.
4,Одерски М.,Scala. Профессиональное программирование. 5-е ...,https://www.piter.com/collection/kompyutery-i-...,799 р.
5,Яворски М.,Python. Лучшие практики и инструменты. 4-е изд.,https://www.piter.com/collection/kompyutery-i-...,3965 р. / 699 р.
6,Керриск М.,Linux API. Исчерпывающее руководство,https://www.piter.com/collection/kompyutery-i-...,3900 р. / 799 р.
7,Форд Н.,Современный подход к программной архитектуре: ...,https://www.piter.com/collection/kompyutery-i-...,699 р.
8,Таненбаум Э. С.,Компьютерные сети. 6-е изд.,https://www.piter.com/collection/kompyutery-i-...,3704 р. / 799 р.
9,Гриффитс Д.,Head First. Программирование для Android на Ko...,https://www.piter.com/collection/kompyutery-i-...,3690 р. / 699 р.


In [2]:
# Обобщим код на все страницы

# создаем структуру для хранения
dict_df = {
    'author': [],
    'title': [],
    'url_book': [],
    'price': []
          }

In [ ]:
import time

In [3]:
#циклом генерируем номера страниц и составляем правильную сслыку
for page in range(1,4):
    link = 'https://www.piter.com/collection/kompyutery-i-internet?page='+str(page) #правильная ссылка
    time.sleep(5)
    answer = requests.get(link)
    soup = BeautifulSoup(answer.text, 'lxml')
    list_books = soup.find_all('div', attrs={'class': ['grid-3 prod-block book-block',
                                                  'grid-3 prod-block book-block clear-class2 clear-class4',
                                                  'grid-3 prod-block book-block clear-class2']
                                        }
                          )
    for book in list_books:
        dict_df['author'].append(book.find('span', attrs={'class': 'author'}).get_text(strip=True))
        dict_df['title'].append(book.find('span', attrs={'class': 'title'}).get_text(strip=True))
        dict_df['url_book'].append('https://www.piter.com' + book.find('a').get('href'))
        price = book.find_all('div', attrs={'class': 'grid-5 padded-left'})
        pr = [price[i].find('span', attrs={'class': 'price'}).get_text(strip=True) for i in range(0,len(price))]
        if len(pr) != 0:
            dict_df['price'].append(' / '.join(pr))
        else:
            dict_df['price'].append(np.nan)

In [4]:
pd.DataFrame(dict_df)

,author,title,url_book,price
0,Грегг Б.,Производительность систем,https://www.piter.com/collection/kompyutery-i-...,5199 р. / 799 р.
1,Апельцин Л.,Data Science в действии,https://www.piter.com/collection/kompyutery-i-...,5029 р. / 699 р.
2,Одерски М.,Scala. Профессиональное программирование. 4-е ...,https://www.piter.com/collection/kompyutery-i-...,4436 р. / 799 р.
3,Одерски М.,Scala. Профессиональное программирование. 5-е ...,https://www.piter.com/collection/kompyutery-i-...,799 р.
4,Яворски М.,Python. Лучшие практики и инструменты. 4-е изд.,https://www.piter.com/collection/kompyutery-i-...,3965 р. / 699 р.
5,Керриск М.,Linux API. Исчерпывающее руководство,https://www.piter.com/collection/kompyutery-i-...,3900 р. / 799 р.
6,Форд Н.,Современный подход к программной архитектуре: ...,https://www.piter.com/collection/kompyutery-i-...,3827 р. / 699 р.
7,Таненбаум Э. С.,Компьютерные сети. 6-е изд.,https://www.piter.com/collection/kompyutery-i-...,3704 р. / 799 р.
8,Гриффитс Д.,Head First. Программирование для Android на Ko...,https://www.piter.com/collection/kompyutery-i-...,3690 р. / 699 р.
9,Волкманн М.,Svelte и Sapper в действии,https://www.piter.com/collection/kompyutery-i-...,3492 р. / 699 р.


In [54]:
import requests
from bs4 import BeautifulSoup

In [55]:
link = 'https://www.litres.ru/popular/'
req = requests.get(link)

In [56]:
req.status_code

200

In [57]:
soup = BeautifulSoup(req.text, 'lxml')

In [58]:
soup.prettify()

'<!DOCTYPE html>\n<html>\n <head>\n  <meta charset="utf-8"/>\n  <title>\n   Подтвердите, что вы не робот\n  </title>\n  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>\n  <script async="" defer="" src="https://www.google.com/recaptcha/api.js">\n  </script>\n  <link href="/static/wrapper/css/reset.css?v1" rel="stylesheet" type="text/css"/>\n  <link href="/static/litres/css/style.css?v1" rel="stylesheet" type="text/css"/>\n  <link href="/static/captcha/css/styles.css?v1" rel="stylesheet" type="text/css"/>\n </head>\n <body>\n  <div class="litres_captcha_page">\n   <div class="litres_header">\n    <div class="logo_block">\n     <a href="/">\n      <span class="" title="ЛитРес">\n       ЛитРес\n      </span>\n     </a>\n    </div>\n   </div>\n   <div class="captcha_content">\n    <div class="captcha_picture">\n     <span class="captcha_icon">\n     </span>\n    </div>\n    <div class="captcha_head">\n     Чтобы продолжить,\n     <br/>\n     подтвердите, что вы не ро

In [11]:
list_book = soup.find_all('div', attrs={'class': 'ArtsGrid-module__artWrapper_1j1xJ'})
len(list_book)

32

In [53]:
soup_book

<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>Подтвердите, что вы не робот</title>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<script async="" defer="" src="https://www.google.com/recaptcha/api.js"></script>
<link href="/static/wrapper/css/reset.css?v1" rel="stylesheet" type="text/css"/>
<link href="/static/litres/css/style.css?v1" rel="stylesheet" type="text/css"/>
<link href="/static/captcha/css/styles.css?v1" rel="stylesheet" type="text/css"/></head>
<body>
<div class="litres_captcha_page">
<div class="litres_header">
<div class="logo_block">
<a href="/">
<span class="" title="ЛитРес">ЛитРес</span>
</a>
</div>
</div>
<div class="captcha_content">
<div class="captcha_picture"><span class="captcha_icon"></span></div>
<div class="captcha_head">Чтобы продолжить,<br/>подтвердите, что вы не робот.</div>
<div class="captcha_text">
<p>Мы заметили странную активность с вашего компьютера. Возможно, мы ошиблись, и эта активность идёт не от вас. В так

In [52]:
print(list_book[0].find('p', attrs={'class': 'ArtInfo-modules__title_2T1yc'}).get_text(strip=True))
print(list_book[0].find('a', attrs={'class': 'ArtInfo-modules__author_235kc'}).get_text(strip=True))
print(list_book[0].find('div', attrs={'class': 'ArtRating-module__votes_2zVnA'}).get_text(strip=True))
url_book = 'https://www.litres.ru'+ list_book[0].find('a').get('href')
page_book = requests.get(url_book)
soup_book = BeautifulSoup(page_book.text, 'lxml')
price = soup_book.find('span', attrs={'class': 'new-price'}).get_text(strip=True)
#float(price.split("\xa0")[0].replace(',', '.'))

Практический интеллект. Как критически мыслить, моделировать ситуации, глубоко анализировать и никогда не обманываться
Патрик Кинг
55


AttributeError: ignored

In [48]:
dict_df={'название книги':[],
         'автор книги':[],
         'рейтинг книги':[],
         'цена книги':[]}

In [49]:
price_book = 0
for book in list_book[:4]:
    dict_df['название книги'].append(book.find('p', attrs={'class': 'ArtInfo-modules__title_2T1yc'}).get_text(strip=True))
    dict_df['автор книги'].append(book.find('a', attrs={'class': 'ArtInfo-modules__author_235kc'}).get_text(strip=True))
    dict_df['рейтинг книги'].append(book.find('div', attrs={'class': 'ArtRating-module__votes_2zVnA'}).get_text(strip=True))
    url_book = 'https://www.litres.ru'+ book.find('a').get('href')
    time.sleep(60)
    page_book = requests.get(url_book)
    print(url_book)
    soup_book = BeautifulSoup(page_book.text, 'lxml')
    try:
        price = soup_book.find('span', attrs={'class': 'new-price'}).get_text(strip=True)
        price_book = float(price.split("\xa0")[0].replace(',', '.'))
    except:
        price_book = np.nan
    dict_df['цена книги'].append(price_book)

https://www.litres.ru/book/patrik-king/prakticheskiy-intellekt-kak-kriticheski-myslit-modelirovat-sit-66077544/
https://www.litres.ru/book/anne-dar/rodnaya-krov-67066479/
https://www.litres.ru/book/vyacheslav-petrovich-kanaev/paradoksalnaya-logika-bytiya-69728707/
https://www.litres.ru/book/i-v-revina/advokat-v-ugolovnom-processe-67333373/


In [50]:
pd.DataFrame(dict_df)

,название книги,автор книги,рейтинг книги,цена книги
0,Практический интеллект. Как критически мыслить...,Патрик Кинг,55,NaN
1,Родная кровь,Anne Dar,7515,NaN
2,Парадоксальная логика бытия,Вячеслав Петрович Канаев,6,NaN
3,Адвокат в уголовном процессе,Н. Д. Эриашвили,2,NaN
